# Member 1 — Shared / Common Code + Classification Data Cleaning

This notebook demonstrates and verifies all Member 1 deliverables:
1. **`src/common/data_loader.py`** — Loaders for the 3 datasets
2. **`src/common/preprocessing.py`** — Shared preprocessing (remove_duplicates, handle_missing, handle_outliers, scale_features)
3. **`src/common/visualization.py`** — Shared plots (correlation heatmap, distribution, predicted vs actual)
4. **`src/classification/clean.py`** — NSL-KDD stateless cleaning (clean_nsl_kdd, get_feature_types, summarize_clean)

**Output:** `data/processed/nsl_kdd_clean.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import pandas as pd
import numpy as np
import config

## 1. Data Loading — `src/common/data_loader.py`

In [ ]:
from src.common.data_loader import load_nsl_kdd, load_student, load_online_retail

df_nsl = load_nsl_kdd()
df_stu = load_student()
df_ret = load_online_retail()

In [ ]:
print(f"NSL-KDD:  {df_nsl.shape}, target='{config.NSL_KDD_TARGET}' present: {'label' in df_nsl.columns}")
print(f"Student:  {df_stu.shape}, target='{config.STUDENT_TARGET}' present: {config.STUDENT_TARGET in df_stu.columns}")
print(f"Retail:   {df_ret.shape}, InvoiceNo present: {'InvoiceNo' in df_ret.columns}")

## 2. Shared Preprocessing — `src/common/preprocessing.py`

### 2.1 `remove_duplicates` (stateless — safe before split)

In [ ]:
from src.common.preprocessing import remove_duplicates, handle_missing, handle_outliers, scale_features

print("=== Duplicates in each dataset ===")
for name, df in [("NSL-KDD", df_nsl), ("Student", df_stu), ("Retail", df_ret)]:
    before = len(df)
    cleaned = remove_duplicates(df)
    print(f"  {name}: {before} -> {len(cleaned)} (removed {before - len(cleaned)})")
    print()

### 2.2 `handle_missing` (FITTED — train only)

In [ ]:
# Demo on a small test DataFrame
test_df = pd.DataFrame({"a": [1, 2, None, 4], "b": ["x", None, "y", "z"]})
print("Before:")
print(test_df)

result = handle_missing(test_df)
print("\nAfter (fit):")
print(result)
print(f"Fitted stats: {result.attrs['fitted_stats']}")

# Transform new data with the same stats
test_df2 = pd.DataFrame({"a": [None, 5], "b": ["w", None]})
result2 = handle_missing(test_df2, stats=result.attrs["fitted_stats"])
print("\nTransform (new data):")
print(result2)

### 2.3 `handle_outliers` (FITTED — train only, caps not drops)

In [ ]:
test_df3 = pd.DataFrame({"x": [1, 2, 3, 4, 5, 100]})
result3 = handle_outliers(test_df3, columns=["x"])
print(f"Before: {list(test_df3['x'])}")
print(f"After:  {list(result3['x'])}")
print(f"Caps:   {result3.attrs['fitted_stats']}")

### 2.4 `scale_features` (FITTED — train only)

In [ ]:
test_df4 = pd.DataFrame({"v": [10.0, 20.0, 30.0, 40.0, 50.0]})

res_std = scale_features(test_df4, columns=["v"], method="standard")
print(f"Standard: {list(res_std['v'].round(4))}")

res_mm = scale_features(test_df4, columns=["v"], method="minmax")
print(f"MinMax:   {list(res_mm['v'].round(4))}")
assert list(res_mm['v'].round(4)) == [0.0, 0.25, 0.5, 0.75, 1.0], "MinMax scaling bug!"

## 3. Shared Visualization — `src/common/visualization.py`

In [ ]:
import matplotlib
matplotlib.use("Agg")
from src.common.visualization import plot_correlation_heatmap, plot_distribution, plot_predicted_vs_actual

FIGURES = str(config.FIGURES_DIR)
os.makedirs(FIGURES, exist_ok=True)

In [ ]:
# Correlation heatmap — Student Performance
df_stu_clean = remove_duplicates(df_stu)
if "Extracurricular Activities" in df_stu_clean.columns:
    df_stu_clean["Extracurricular Activities"] = df_stu_clean["Extracurricular Activities"].map({"Yes": 1, "No": 0})
plot_correlation_heatmap(df_stu_clean, save_path=f"{FIGURES}/p1_corr_heatmap_student.png")

In [ ]:
# Distribution of Performance Index (numeric)
plot_distribution(df_stu_clean, "Performance Index", save_path=f"{FIGURES}/p1_distribution_performance_index.png")

In [ ]:
# Distribution of label on NSL-KDD (categorical — shows class imbalance)
df_nsl_clean = remove_duplicates(df_nsl)
plot_distribution(df_nsl_clean, "label", save_path=f"{FIGURES}/p1_distribution_nsl_label.png")

In [ ]:
# Predicted vs actual demo (throwaway linear model)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_stu = df_stu_clean.drop(columns=["Performance Index"]).select_dtypes(include=[np.number])
y_stu = df_stu_clean["Performance Index"]
X_tr, X_te, y_tr, y_te = train_test_split(X_stu, y_stu, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_tr, y_tr)
y_pred = lr.predict(X_te)
plot_predicted_vs_actual(y_te, y_pred, "Demo: Linear Regression on Student Data",
                         save_path=f"{FIGURES}/p1_predicted_vs_actual_demo.png")

## 4. Classification Data Cleaning — `src/classification/clean.py`

Stateless cleaning of NSL-KDD: remove duplicates, fix dtypes, drop `difficulty_level`, separate target.

In [ ]:
from src.classification.clean import clean_nsl_kdd, get_feature_types, summarize_clean

# Load raw and clean
df_raw = load_nsl_kdd()
X_clean, y = clean_nsl_kdd(df_raw)

In [ ]:
# Feature types for Member 2
cat_cols, num_cols = get_feature_types(X_clean)
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numeric columns ({len(num_cols)}): {num_cols[:5]} ... ({len(num_cols)} total)")

In [ ]:
# Full summary
summary = summarize_clean(X_clean, y)
summary

In [ ]:
# Verify the saved clean file
df_check = pd.read_csv(config.CLEAN_NSL_KDD)
print(f"Saved file shape: {df_check.shape}")
print(f"Columns: {list(df_check.columns[:5])} ... {list(df_check.columns[-3:])}")
print(f"Duplicates: {df_check.duplicated().sum()}")
print(f"'difficulty_level' removed: {'difficulty_level' not in df_check.columns}")
print(f"'label' present: {'label' in df_check.columns}")
print("\n✅ data/processed/nsl_kdd_clean.csv is ready for Member 2 and Member 3!")